# Extraction Basics

This notebook demonstrates the core extraction pipeline:
**Text → LLM (Instructor + Ollama) → Pydantic Objects**

We test extraction at increasing complexity levels to understand where LLMs succeed and where they struggle.

## Prerequisites
1. Ollama running locally: `ollama serve`
2. Model pulled: `ollama pull llama3.2`
3. Dependencies installed: `uv sync`

## Setup

In [ ]:
from backend.extraction.extractor import Extractor
from backend.schemas.process_knowledge.entities import (
    Tool, Material, PPE, Worker, Step, Procedure,
    ProcessParameter, QualityRequirement
)
from backend.schemas.process_knowledge.relations import (
    StepOrder, ToolRequirement, StepExecution,
    QualityCheck, ProcedureExecution, StepFeedback
)
from backend.schemas.process_knowledge.examples import (
    LEVEL_1_TEXT, LEVEL_1_EXPECTED_ENTITIES,
    LEVEL_2_TEXT, LEVEL_2_EXPECTED_PROCEDURE, LEVEL_2_EXPECTED_STEPS,
    LEVEL_3_TEXT, LEVEL_3_EXPECTED,
    LEVEL_4_TEXT, LEVEL_4_EXPECTED,
    LEVEL_5_TEXT, LEVEL_5_EXPECTED
)

extractor = Extractor(model="llama3.2", max_retries=1)

---
## Level 1 — Flat Entities

**Task:** Extract simple entities (Tool, Material) from a short text.
This should be easy for any LLM.

In [ ]:
print(LEVEL_1_TEXT.strip())

In [ ]:
tools = extractor.extract_list(LEVEL_1_TEXT, Tool)

print("EXTRACTED TOOLS:")
for tool in tools:
    print(tool.model_dump_json(indent=2))

In [ ]:
materials = extractor.extract_list(LEVEL_1_TEXT, Material)

print("EXTRACTED MATERIALS:")
for mat in materials:
    print(mat.model_dump_json(indent=2))

In [ ]:
print("EXPECTED (Gold Standard):")
for entity in LEVEL_1_EXPECTED_ENTITIES:
    print(f"  {type(entity).__name__}: {entity.name}")

**Questions to explore:**
- Did the LLM find all entities?
- Are the field values correct (manufacturer, specification, etc.)?
- Did it hallucinate anything not in the text?

## EVALUATION

In [ ]:
from backend.evaluation.evaluators import evaluate_entity_detection
from backend.evaluation.matching import MatchStrategy
from backend.evaluation.metrics import format_prf1

# Evaluate Level 1: Did we find the right Tools?
tools = extractor.extract_list(LEVEL_1_TEXT, Tool)
metrics, log = evaluate_entity_detection(
    LEVEL_1_EXPECTED_ENTITIES, tools,
    match_strategy=MatchStrategy.TOKEN_OVERLAP,
    match_threshold=0.6
)

print(format_prf1(metrics))
print("\nDetailed log:")
for entry in log:
    print(f"  {entry['status']}: expected={entry['expected']} → extracted={entry['extracted']} (score={entry['score']})")

# Same evaluation but with different matching strategy
for strategy in MatchStrategy:
    if strategy == MatchStrategy.EMBEDDING:
        continue  # skip if sentence-transformers not installed
    metrics, _ = evaluate_entity_detection(
        LEVEL_1_EXPECTED_ENTITIES, tools,
        match_strategy=strategy
    )
    print(f"  {strategy.value:20s} → P={metrics.precision:.3f} R={metrics.recall:.3f} F1={metrics.f1:.3f}")

---
## Level 2 — Binary Relations

**Task:** Extract a procedure with its steps and their ordering.
The LLM must recognize sequence ("zuerst... dann... anschließend...").

In [ ]:
print(LEVEL_2_TEXT.strip())

In [ ]:
procedure = extractor.extract(LEVEL_2_TEXT, Procedure)

print("EXTRACTED PROCEDURE:")
print(procedure.model_dump_json(indent=2))

In [ ]:
steps = extractor.extract_list(LEVEL_2_TEXT, Step)

print("EXTRACTED STEPS:")
for step in steps:
    print(f"  #{step.step_number}: {step.name} ({step.step_type})")

In [ ]:
try:
    step_orders = extractor.extract_list(LEVEL_2_TEXT, StepOrder)
    print("EXTRACTED STEP ORDERING:")
    for order in step_orders:
        print(f"  {order.before.name}  →  {order.after.name}")
except Exception as e:
    print(f"⚠️ StepOrder extraction failed: {type(e).__name__}")
    print(f"   Could be expected as nested relations are hard for small models.")
    print(f"   Try with a larger model: Extractor(model='llama3.1:70b')")

**Questions to explore:**
- Did it correctly assign step numbers?
- Is the ordering complete (all sequential pairs)?
- Did it get `is_critical` right for any step?

---
## Level 3 — Relations with Attributes

**Task:** Extract a tool requirement with nested process parameters.
The relation itself carries data (configuration, parameters).

In [ ]:
print(LEVEL_3_TEXT.strip())

In [ ]:
tool_req = extractor.extract(LEVEL_3_TEXT, ToolRequirement)

print("EXTRACTED TOOL REQUIREMENT:")
print(tool_req.model_dump_json(indent=2))

In [ ]:
print("PARAMETER CHECK:")
if tool_req.parameters:
    for param in tool_req.parameters:
        print(f"  {param.name}: {param.nominal_value} {param.unit} (type: {param.parameter_type})")
else:
    print("  ⚠️ No parameters extracted!")

**Questions to explore:**
- Were all three parameters extracted (current, voltage, wire speed)?
- Are the units correct?
- Did it capture the configuration note about the contact tip?

---
## Level 4 — N-ary Relations

**Task:** Extract a step execution — who did what, when, how, with what deviations.
This is a complex n-ary relation with multiple participants and attributes.

In [ ]:
print(LEVEL_4_TEXT.strip())

In [ ]:
step_exec = extractor.extract(LEVEL_4_TEXT, StepExecution)

print("EXTRACTED STEP EXECUTION:")
print(step_exec.model_dump_json(indent=2))

In [ ]:
print("KEY FIELD CHECK:")
print(f"  Step: {step_exec.step.name}")
print(f"  Worker: {step_exec.executed_by.name} ({step_exec.executed_by.role})")
print(f"  Status: {step_exec.status}")
print(f"  Duration: {step_exec.actual_duration_seconds}s")
print(f"  Deviation: {step_exec.deviation_from_spec}")
if step_exec.tools_used:
    print(f"  Tools: {[t.name for t in step_exec.tools_used]}")
if step_exec.observed_parameters:
    for p in step_exec.observed_parameters:
        print(f"  Parameter: {p.name} = {p.nominal_value} {p.unit}")

**Questions to explore:**
- Did it extract the datetime correctly?
- Did it capture the deviation (185A vs 180A)?
- Did it calculate duration (12 minutes = 720 seconds)?
- Are the nested objects (Worker, Step, Tool) properly filled?

---
## Level 5 — Deeply Nested Structures

**Task:** Extract a complete procedure execution with nested step executions,
quality checks, and feedback. This is the hardest test.

⚠️ **This is where LLMs typically start to struggle.** Watch for:
- Missing step executions
- Wrong worker assignments
- Lost feedback/quality check details
- Hallucinated information

In [ ]:
print(LEVEL_5_TEXT.strip())

In [ ]:
proc_exec = extractor.extract(LEVEL_5_TEXT, ProcedureExecution)

print("EXTRACTED PROCEDURE EXECUTION:")
print(proc_exec.model_dump_json(indent=2))

In [ ]:
print("EXTRACTION SUMMARY:")
print(f"  Procedure: {proc_exec.procedure.name} ({proc_exec.procedure.procedure_id})")
print(f"  Workers: {[w.name for w in proc_exec.executed_by]}")
print(f"  Supervisor: {proc_exec.supervised_by.name if proc_exec.supervised_by else 'None'}")
print(f"  Overall status: {proc_exec.overall_status}")
print(f"  Overall result: {proc_exec.overall_result}")

print(f"\n  Step Executions: {len(proc_exec.step_executions) if proc_exec.step_executions else 0}")
if proc_exec.step_executions:
    for se in proc_exec.step_executions:
        deviation = f" ⚠️ {se.deviation_from_spec}" if se.deviation_from_spec else ""
        print(f"    #{se.step.step_number}: {se.step.name} ({se.status}) - {se.actual_duration_seconds}s{deviation}")

print(f"\n  Quality Checks: {len(proc_exec.quality_checks) if proc_exec.quality_checks else 0}")
if proc_exec.quality_checks:
    for qc in proc_exec.quality_checks:
        print(f"    Inspector: {qc.inspector.name}, Result: {qc.result}, Findings: {qc.findings}")

print(f"\n  Feedback: {len(proc_exec.feedback) if proc_exec.feedback else 0}")
if proc_exec.feedback:
    for fb in proc_exec.feedback:
        print(f"    {fb.reported_by.name} ({fb.feedback_type}): {fb.content}")
        if fb.resolved:
            print(f"      → Resolved by {fb.resolved_by.name if fb.resolved_by else '?'}: {fb.resolution}")

---
## Comparison: Expected vs. Extracted

In [ ]:
expected = LEVEL_5_EXPECTED

print("EXPECTED vs EXTRACTED — Level 5")
print("=" * 50)

exp_steps = len(expected.step_executions) if expected.step_executions else 0
got_steps = len(proc_exec.step_executions) if proc_exec.step_executions else 0
print(f"  Step Executions:  expected={exp_steps}, got={got_steps}  {'✅' if exp_steps == got_steps else '❌'}")

exp_qc = len(expected.quality_checks) if expected.quality_checks else 0
got_qc = len(proc_exec.quality_checks) if proc_exec.quality_checks else 0
print(f"  Quality Checks:   expected={exp_qc}, got={got_qc}  {'✅' if exp_qc == got_qc else '❌'}")

exp_fb = len(expected.feedback) if expected.feedback else 0
got_fb = len(proc_exec.feedback) if proc_exec.feedback else 0
print(f"  Feedback:         expected={exp_fb}, got={got_fb}  {'✅' if exp_fb == got_fb else '❌'}")

exp_workers = len(expected.executed_by)
got_workers = len(proc_exec.executed_by)
print(f"  Workers:          expected={exp_workers}, got={got_workers}  {'✅' if exp_workers == got_workers else '❌'}")

print(f"  Supervisor:       expected={expected.supervised_by.name if expected.supervised_by else 'None'}, got={proc_exec.supervised_by.name if proc_exec.supervised_by else 'None'}")
print(f"  Overall Status:   expected={expected.overall_status}, got={proc_exec.overall_status}")
print(f"  Overall Result:   expected={expected.overall_result}, got={proc_exec.overall_result}")

---
## Next Steps

Now that you've seen the basic extraction pipeline, here's what to explore:

1. **Try different models:** Change `Extractor(model="llama3.2")` to other Ollama models. Does a bigger model improve Level 5 extraction?

2. **Try custom prompts:** Pass `system_prompt=` to `extract()` with more specific instructions. Does few-shot prompting (with examples) help?

3. **Try the organizational knowledge domain:** Use the schemas from `backend.schemas.organizational_knowledge` with your own test texts.

4. **Break it:** Write texts that are ambiguous, contradictory, or incomplete. How does the extractor handle edge cases?

5. **Measure it:** Build evaluation functions that compare extracted vs. expected and compute Precision/Recall/F1 per entity type and complexity level.